<a href="https://colab.research.google.com/github/Aadit-5810228/Aadit-5810228-AnalyticsEngineering/blob/main/2_03_practice_bike_hire_student%20v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

![](https://drive.google.com/uc?export=view&id=17M9oGaOj4KdKiDDH3oTibxJ9qBTSjp6n)

# 2.03 Practice session
## Bike-hire trips

## How this practice works

This is a **short rehearsal** of the timed test. You do not need to finish or get the "right" answer. The aim is to practise the way of working.

- You have about **45 minutes**. Work individually.
- You may use **any AI tool** (for example Gemini in Colab). We are **not marking** anything today. In the real test, the quality of AI-generated code will not matter. What matters is the control youDshow.
- Write in markdown cells as you go: your thinking, **each prompt you send (paste it as you send it)**, and the checks you make.
- Unfinished work is fine. Say what you did not get to.
- At the end, download the Notebook (using File > Download > Download .ipynb). Once it has completed please zip the file up. This is the artefact you would submit for the in-class test.
- You can also save a copy to your GitHub using the instructions from my.wbs (1b asychronous tasks).

**Suggested timing**

| Minutes | Focus |
| --- | --- |
| 0 to 10 | Look at the tables and note what you find |
| 10 to 25 | Clean and prepare the data |
| 25 to 38 | Join and aggregate to answer the question |
| 38 to 45 | Check your answer and write up |

At the end, use **Runtime, then Restart session and run all** and note anything that fails.


## The data

A fictional bike-hire scheme: three tables covering trips taken in March 2025.

| Table | What it holds |
| --- | --- |
| `trips` | One record per trip: bike, start and end station, start time, duration and user type |
| `bikes` | The bikes, with type and purchase year |
| `stations` | The docking stations, with zone and capacity |

**How they connect:** `trips.bike_id` to `bikes.bike_id`, and `trips.start_station_id` (and `end_station_id`) to `stations.station_id`.

Check the real column names in the next cell before you write anything.


In [2]:
# ---- Setup: run this cell first ----
!pip install -q duckdb

import glob
import os

import duckdb
import pandas as pd

DATA_URL = "https://github.com/MJMortensonWarwick/adminaeaiv/releases/download/dbsession/bike-hire-practice.zip"

DATA_DIR = "/content/bikes"
# Download only if no CSV files are there yet. The CSVs may sit inside a folder
# within the zip; the search below looks in sub-folders too.
if DATA_URL and not glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True):
    import urllib.request
    import zipfile
    os.makedirs(DATA_DIR, exist_ok=True)
    urllib.request.urlretrieve(DATA_URL, "/content/bikes.zip")
    zipfile.ZipFile("/content/bikes.zip").extractall(DATA_DIR)

FILES = {"trips": "trips.csv", "bikes": "bikes.csv", "stations": "stations.csv"}

tables, missing = {}, []
for name, filename in FILES.items():
    found = glob.glob(os.path.join(DATA_DIR, "**", filename), recursive=True)
    if found:
        tables[name] = pd.read_csv(found[0])
    else:
        missing.append(filename)

if missing:
    found_csvs = sorted(os.path.basename(f) for f in glob.glob(os.path.join(DATA_DIR, "**", "*.csv"), recursive=True))
    raise FileNotFoundError(
        f"Missing files: {missing}. CSV files found under {DATA_DIR}: {found_csvs}. Ask for help."
    )

# Make every table available as a DuckDB table, so you can use SQL or pandas.
con = duckdb.connect()
for name, df in tables.items():
    con.register(name, df)


def q(sql):
    """Run SQL against the tables above and return a pandas DataFrame."""
    return con.sql(sql).df()


for name, df in tables.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")
    print("   ", list(df.columns))


trips: 612 rows, 7 columns
    ['trip_id', 'bike_id', 'start_station_id', 'end_station_id', 'start_time', 'duration_min', 'user_type']
bikes: 60 rows, 3 columns
    ['bike_id', 'bike_type', 'purchase_year']
stations: 12 rows, 4 columns
    ['station_id', 'station_name', 'zone', 'capacity']


## Examples: two ways to work with the tables

You can use SQL, pandas, or a mix of both. The next two cells show each. Run them and change them to see what happens. You do not have to use them in your answer.


In [ ]:
# Example 1: SQL
# q() runs SQL against any of the tables above and returns a pandas DataFrame.
from IPython.display import display

display(q("SELECT * FROM trips LIMIT 5"))

display(q("SELECT COUNT(*) AS n_rows, COUNT(DISTINCT trip_id) AS n_distinct_trips FROM trips"))

display(q("""
    SELECT t.trip_id, s.station_name
    FROM trips AS t
    JOIN stations AS s ON t.start_station_id = s.station_id
    LIMIT 5
"""))


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


,n_rows,n_distinct_trips
0,612,600


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


In [ ]:
# Example 2: pandas
# Each table is also a pandas DataFrame, stored in the dictionary `tables`.
from IPython.display import display

trips = tables["trips"]
display(trips.head())
print(trips.shape)
print(trips.dtypes)

joined = tables["trips"].merge(tables["stations"], left_on="start_station_id", right_on="station_id", how="inner")
display(joined[["trip_id", "station_name"]].head())

# You can mix the two: run SQL, then use pandas methods on the result
result = q("SELECT start_station_id FROM trips")
print(result["start_station_id"].nunique(), "different start stations in trips")


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member


(612, 7)
trip_id              object
bike_id              object
start_station_id     object
end_station_id       object
start_time           object
duration_min        float64
user_type            object
dtype: object


,trip_id,station_name
0,T0001,Park Road
1,T0002,Cathedral Close
2,T0003,Sports Centre
3,T0004,University Gate
4,T0005,Market Square


12 different start stations in trips


## Your brief

**From:** the scheme manager

> We want to know whether electric bikes are used for longer trips than standard bikes, and whether that differs by zone. Please give us the **average trip duration by bike type and by the zone of the start station**.

![](https://drive.google.com/uc?export=view&id=1eH5Q3l4LnzwZ-XcT_kJDeHoKA2rufiOW)

**What your final table should look like**

One row per combination of bike type and zone, with the number of trips and the average duration in minutes. State what one row represents.

**Things to think about** (you do not have to solve all of them)

- Are there missing, duplicated or odd values in the tables you use? What will you do about each, and why?
- Do the bike types and dates look consistent?
- How will you check your final table is believable?


## 1. Look at the data

*What does one row of each table represent? Which columns will you use? What looks odd, missing or inconsistent? Write down what you find.*

First I will be taking a quick glance over the three tables. This is to mainly understand the overall data i will be working with and spot any key sources of errors I could fix during the cleaning and checking process.

Prompt: First loop through all the three data sets - bikes, trips and stations and give me the first 5 rows of each of them. Also tell the number rows in each data set. What I am expecting is an output of table heads and number of rows. Show a clear seperatation between each table so that it looks clean

In [3]:
for name, df in tables.items():
    print("=" * 50)
    print(f"Table: {name.upper()} ({df.shape[0]} rows)")
    print("=" * 50)
    display(df.head(5))
    print("\n")

Table: TRIPS (612 rows)


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
0,T0001,B028,S05,S01,2025-03-01 05:52:00,21.0,Casual
1,T0002,B043,S12,S12,2025-03-01 06:31:00,27.0,CASUAL
2,T0003,B045,S11,S07,2025-03-01 07:06:00,18.0,Casual
3,T0004,B051,S04,S04,2025-03-01 07:06:00,34.0,Member
4,T0005,B049,S01,S09,2025-03-01 07:36:00,16.0,Member




Table: BIKES (60 rows)


,bike_id,bike_type,purchase_year
0,B001,Standard,2021
1,B002,Electric,2022
2,B003,electric,2024
3,B004,Standard,2021
4,B005,STANDARD,2022




Table: STATIONS (12 rows)


,station_id,station_name,zone,capacity
0,S01,Market Square,Centre,29.0
1,S02,Railway Station,Centre,23.0
2,S03,Canal Basin,North,24.0
3,S04,University Gate,West,28.0
4,S05,Park Road,South,22.0


Okay so from this output I already spot a few errors that I need to clean up. I need to ensure there is no duplication in any of the ID columns since they are the primary keys. Any column with data time needs to be consistent. The categorical data such as the station name or user type needs to be in consistent formatting to avoid duplicates. Lastly, the I also need to do outlier checks for all the quantitative data.

I will checking and cleaning the data sets simaltaneously.

First the Trips data -

In [6]:
# Prompt: Check for any NA or Duplicate values in the trips data set and if there are any NA or duplicates then list them
trips_df = tables["trips"]

print("=== Missing (NA) Values in Trips ===")
print(trips_df.isnull().sum())

# Filter and display the specific rows with NA duration
na_durations = trips_df[trips_df["duration_min"].isnull()]
print(f"\nTotal rows with NA duration: {len(na_durations)}")
if len(na_durations) > 0:
    print("\nRows with NA duration:")
    display(na_durations)

print("\n=== Duplicate Rows in Trips ===")
num_duplicates = trips_df.duplicated().sum()
print(f"Total duplicate rows: {num_duplicates}")

if num_duplicates > 0:
    print("\nDuplicate rows:")
    display(trips_df[trips_df.duplicated(keep=False)])

print("\n=== Duplicate Trip IDs (Primary Key check) ===")
num_id_duplicates = trips_df["trip_id"].duplicated().sum()
print(f"Total duplicate trip_ids: {num_id_duplicates}")

=== Missing (NA) Values in Trips ===
trip_id              0
bike_id              0
start_station_id     0
end_station_id       0
start_time           0
duration_min        30
user_type            0
dtype: int64

Total rows with NA duration: 30

Rows with NA duration:


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
12,T0013,B024,S04,S02,01/03/2025 18:43,NaN,Member
14,T0015,B040,S03,S04,2025-03-01 19:03:00,NaN,Member
30,T0031,B015,S06,S06,2025-03-02 08:50:00,NaN,Casual
60,T0061,B011,S09,S07,2025-03-04 08:23:00,NaN,Member
64,T0065,B041,S06,S02,04/03/2025 09:06,NaN,Member
73,T0074,B016,S06,S02,04/03/2025 17:05,NaN,Member
86,T0087,B012,S04,S04,2025-03-05 12:27:00,NaN,Casual
118,T0118,B025,S06,S12,2025-03-07 06:27:00,NaN,Member
124,T0124,B036,S01,S06,2025-03-07 08:39:00,NaN,Casual
146,T0146,B031,S12,S12,2025-03-08 18:45:00,NaN,Member



=== Duplicate Rows in Trips ===
Total duplicate rows: 12

Duplicate rows:


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
96,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
97,T0097,B031,S12,S12,2025-03-06 08:14:00,10.0,Casual
175,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
176,T0175,B059,S04,S09,2025-03-09 21:51:00,13.0,Casual
185,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
186,T0184,B034,S02,S03,2025-03-10 12:23:00,25.0,Casual
285,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
286,T0283,B014,S07,S04,2025-03-15 19:48:00,5.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member



=== Duplicate Trip IDs (Primary Key check) ===
Total duplicate trip_ids: 12


At the silver level we can keep the NA values in the duration column considering in the future we need to aggregate to get  numbers on the category of the member or any such exploration. However these NA values will be removed during the gold layer of this investigation.

Next we need to check for consistent formatting in the start_time column

Prompt: Loop through each of the values in the start_time column in the trips data set and ensure they pass a consistency check. The date format needs to be ISO standard. If they are not consistent then loop through them to fix the format.

In [9]:
import pandas as pd
import glob
import os

# Safely re-load a fresh copy of the trips dataset from the original file to reverse earlier mutations
csv_path = glob.glob(os.path.join("/content/bikes", "**", "trips.csv"), recursive=True)[0]
fresh_trips_df = pd.read_csv(csv_path)

# Define the target ISO format pattern: YYYY-MM-DD HH:MM:SS
iso_pattern = r'^\d{4}-\d{2}-\d{2} \d{2}:\d{2}:\d{2}$'

# Check consistency on the fresh unmutated data
is_consistent = fresh_trips_df["start_time"].astype(str).str.match(iso_pattern)
inconsistent_entries = fresh_trips_df[~is_consistent]

print(f"Total trips checked (fresh copy): {len(fresh_trips_df)}")
print(f"Consistent with ISO format (YYYY-MM-DD HH:MM:SS): {is_consistent.sum()}")
print(f"Inconsistent/non-ISO format entries detected: {len(inconsistent_entries)}")

if len(inconsistent_entries) > 0:
    print("\nExamples of inconsistent formats in 'start_time':")
    display(inconsistent_entries[["trip_id", "start_time"]].head(10))
else:
    print("\nAll entries are fully consistent with the ISO format!")

Total trips checked (fresh copy): 612
Consistent with ISO format (YYYY-MM-DD HH:MM:SS): 514
Inconsistent/non-ISO format entries detected: 98

Examples of inconsistent formats in 'start_time':


,trip_id,start_time
6,T0007,01/03/2025 09:26
12,T0013,01/03/2025 18:43
15,T0016,01/03/2025 19:16
50,T0051,03/03/2025 08:54
64,T0065,04/03/2025 09:06
66,T0067,04/03/2025 10:47
68,T0069,04/03/2025 11:37
73,T0074,04/03/2025 17:05
83,T0084,05/03/2025 08:32
85,T0086,05/03/2025 09:29


These date-time formats need to fixed as well. Now lastly, we need to check for the outliers in the duration column to exclude any major outliers. I am expecting to run an IQR calculation to find statistical outliers.

Prompt: Using the IQR find the outliers in the duration_min column in the trips data set. List those rows for me to check.

In [11]:

trips_df = tables["trips"]

# Calculate Q1 (25th percentile) and Q3 (75th percentile) for duration_min
Q1 = trips_df["duration_min"].quantile(0.25)
Q3 = trips_df["duration_min"].quantile(0.75)
IQR = Q3 - Q1

# Define bounds for identifying outliers
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# Identify the outlier rows (ignoring NaN values for this calculation)
outliers = trips_df[(trips_df["duration_min"] < lower_bound) | (trips_df["duration_min"] > upper_bound)]

print(f"Q1 (25th percentile): {Q1} min")
print(f"Q3 (75th percentile): {Q3} min")
print(f"IQR: {IQR} min")
print(f"Outlier bounds: Below {lower_bound} or Above {upper_bound}\n")
print(f"Number of statistical outliers found: {len(outliers)}")

if len(outliers) > 0:
    display(outliers.sort_values(by="duration_min", ascending=False))

Q1 (25th percentile): 12.0 min
Q3 (75th percentile): 22.0 min
IQR: 10.0 min
Outlier bounds: Below -3.0 or Above 37.0

Number of statistical outliers found: 20


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
80,T0081,B010,S12,S11,2025-03-04 21:57:00,2880.0,Member
143,T0143,B020,S09,S10,2025-03-08 12:23:00,1440.0,Casual
287,T0284,B046,S08,S03,2025-03-15 20:25:00,1440.0,Member
127,T0127,B025,S08,S09,2025-03-07 09:01:00,90.0,Member
357,T0352,B042,S03,S02,2025-03-19 08:59:00,67.0,Member
326,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
327,T0323,B023,S08,S05,2025-03-17 17:17:00,52.0,Member
520,T0512,B045,S07,S04,NaN,51.0,Member
37,T0038,B016,S12,S09,2025-03-02 12:23:00,50.0,Casual
262,T0260,B025,S04,S04,NaN,46.0,Member


The output suggest that all the values above 37 min are statistical outliers. However, logically if you think trips that lasted for upto 90 min are also plausible. Hence, I will keep all the entries with upto 100 min within the data set. Rests of the values that are above 1000 min dont make logical sense, hence they will be excluded.

Now I will clean the trips data set.

Prompt: Create a copy of the trips data set. Name it silver_trips.

In [12]:
# Create a clean copy of the trips dataset as silver_trips
silver_trips = tables["trips"].copy()
print("Created 'silver_trips' copy.")
print(f"Shape: {silver_trips.shape[0]} rows, {silver_trips.shape[1]} columns")

Created 'silver_trips' copy.
Shape: 612 rows, 7 columns


First I will delete the duplicates, there were 12 of them so I am expecting 600 rows to remain after dropping the duplicates

In [13]:
#Prompt: Now drop the duplicates we found above and give a discription with numeber of rows
silver_trips = silver_trips.drop_duplicates()
print(f"Shape: {silver_trips.shape[0]} rows, {silver_trips.shape[1]} columns")


Shape: 600 rows, 7 columns


Now I will clean the data-time formatting to make it consistent

Prompt: Clean the date-time formatting in the start_time column to make it consistent to format used in the test. Also show the rows that have been changed

In [17]:
# Load fresh trips data to reverse any previous in-memory mutations
import glob
import os

csv_path = glob.glob(os.path.join("/content/bikes", "**", "trips.csv"), recursive=True)[0]
fresh_trips = pd.read_csv(csv_path)

# Re-apply the duplicate removal on fresh data and explicitly use .copy() to avoid SettingWithCopyWarning
silver_trips = fresh_trips.drop_duplicates().copy()

# Keep track of the original start times before parsing
original_start_time = silver_trips["start_time"].copy()

# Parse mixed-format dates and convert to standard ISO format string (YYYY-MM-DD HH:MM:SS)
silver_trips["start_time"] = pd.to_datetime(silver_trips["start_time"], format='mixed', dayfirst=True).dt.strftime('%Y-%m-%d %H:%M:%S')

# Identify changed rows
changed_mask = original_start_time != silver_trips["start_time"]
print(f"Shape: {silver_trips.shape[0]} rows, {silver_trips.shape[1]} columns")
print(f"Number of timestamps reformatted: {changed_mask.sum()}")

if changed_mask.sum() > 0:
    print("\nSample of reformatted rows:")
    display(pd.DataFrame({
        "Original": original_start_time[changed_mask],
        "Cleaned (ISO)": silver_trips["start_time"][changed_mask]
    }).head(10))

Shape: 600 rows, 7 columns
Number of timestamps reformatted: 97

Sample of reformatted rows:


,Original,Cleaned (ISO)
6,01/03/2025 09:26,2025-03-01 09:26:00
12,01/03/2025 18:43,2025-03-01 18:43:00
15,01/03/2025 19:16,2025-03-01 19:16:00
50,03/03/2025 08:54,2025-03-03 08:54:00
64,04/03/2025 09:06,2025-03-04 09:06:00
66,04/03/2025 10:47,2025-03-04 10:47:00
68,04/03/2025 11:37,2025-03-04 11:37:00
73,04/03/2025 17:05,2025-03-04 17:05:00
83,05/03/2025 08:32,2025-03-05 08:32:00
85,05/03/2025 09:29,2025-03-05 09:29:00


Lastly I will fix the extreme values that go above 100 min as duration. I am expected to lose 3 rows which were above 100 min.

Prompt: for the values that go above 100 min in the duration column, drop them.




In [18]:
# Filter out rows where duration is strictly greater than 100 minutes
# Keep rows that are <= 100 or are NaN (since we decided to handle NaNs in a later step)
outliers_mask = silver_trips["duration_min"] > 100
dropped_trips = silver_trips[outliers_mask]

silver_trips = silver_trips[~outliers_mask].copy()

print(f"Original shape before dropping extreme durations: 600 rows")
print(f"New shape: {silver_trips.shape[0]} rows, {silver_trips.shape[1]} columns")
print(f"Number of extreme duration rows removed: {len(dropped_trips)}")

if len(dropped_trips) > 0:
    print(
        "\nDropped rows with duration > 100 min:"
    )
    display(dropped_trips)

Original shape before dropping extreme durations: 600 rows
New shape: 597 rows, 7 columns
Number of extreme duration rows removed: 3

Dropped rows with duration > 100 min:


,trip_id,bike_id,start_station_id,end_station_id,start_time,duration_min,user_type
80,T0081,B010,S12,S11,2025-03-04 21:57:00,2880.0,Member
143,T0143,B020,S09,S10,2025-03-08 12:23:00,1440.0,Casual
287,T0284,B046,S08,S03,2025-03-15 20:25:00,1440.0,Member


Now I will move to the Bike data set

I will be do three main checks here. The bike Ids cant have duplicates. The year formatting has to consistent. And I already know that the formatting in the bike type column is inconsistent so i will be fixing that directly.

First lets start with checking for the duplicates.

Prompt: Check for duplicates in the bikes data set. Check entire rows and bike id column.

In [19]:
bikes_df = tables["bikes"]

# Check for duplicate entire rows
duplicate_rows = bikes_df.duplicated().sum()
print(f"Total duplicate rows in bikes: {duplicate_rows}")

# Check for duplicate bike_ids (primary key check)
duplicate_ids = bikes_df["bike_id"].duplicated().sum()
print(f"Total duplicate bike_ids (primary key): {duplicate_ids}")

if duplicate_ids > 0:
    print("\nDuplicate bike ID entries:")
    display(bikes_df[bikes_df.duplicated(subset=["bike_id"], keep=False)])

Total duplicate rows in bikes: 0
Total duplicate bike_ids (primary key): 0


Now lets check the for consistent formatting in the year column

Prompt: check for unique values in the year column of the bikes data set.

In [20]:
unique_years = sorted(bikes_df["purchase_year"].unique())
print(f"Unique values in purchase_year: {unique_years}")

Unique values in purchase_year: [np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024)]


There are no issues in th year formatting. Now ill directly jump to fixing the formatting on the bile type column. I am expecting each row to be either Standard or Electric.

Prompt: The bike type column in the bikes data set is inconsistent. The bikes can either be "Standard" or "Electric".

In [23]:
# Create silver_bikes copy
silver_bikes = tables["bikes"].copy()

# Standardize bike_type strings by stripping whitespace and converting to title case
silver_bikes["bike_type"] = silver_bikes["bike_type"].str.strip().str.title()

# Map "E-Bike" to "Electric"
silver_bikes["bike_type"] = silver_bikes["bike_type"].replace("E-Bike", "Electric")

# Verify the clean unique values
clean_bike_types = silver_bikes["bike_type"].unique()
print(f"Cleaned unique values in silver_bikes['bike_type']: {clean_bike_types}")
print(f"Value counts:\n{silver_bikes['bike_type'].value_counts()}")

Cleaned unique values in silver_bikes['bike_type']: ['Standard' 'Electric']
Value counts:
bike_type
Standard    47
Electric    13
Name: count, dtype: int64


We still have 60 rows so there was no data loss.

Now we will move to

[link text](https://)## 2. Clean and prepare

*Decide what to do about each problem you found, say why, and do it. Short cells that you can re-run are better than one long cell.*

## 3. Join and aggregate

*Join the tables and calculate the answer. Note how many rows you expect after each join, then check.*

## 4. Check

*How do you know the answer is right? Compare a count, spot check a few records, or recalculate one number a different way. What did you find?*

All the checks were done during the steps above. Depending on the type of data the checking was customized for the column.

## 5. Prompts and AI notes

*Paste each prompt as you sent it. Note what you accepted, changed or rejected and why.*

The prompts used are mentioned in the steps above. The codw and the output was thoroughly checked. I have also added my comments

## 6. What I could not verify

*What remains unchecked? What would you do next?*

I could not complete the data check and cleaning for the stations data set. for this data set I would have made sure there are no duplicate rows or suplicate station ids. I would have also checked and cleaned if the formatting of station name and zone was inconsistent. I would have performed the IQR test to find any outliers in the capcity column. Additionally i would have performed a general NA value test.

I would have merged the data set using the primary keys and perfomed checks for any data loss or inconsistencies.

For the gold standard data, I would have removed the NA values in teh duration_min column to avoid any error during aggregation. My final output would have given teh number of trips and the average duration in minutes by bike types and zone.